# Smart Currency AI V2 — Data, Augmentation, Training & Evaluation

This notebook is designed for the project folder structure in `Smart-Currency-AI-V2`.

## Uploads required when prompted
1. `original_images.zip`: contains folders `10`, `20`, `50_front`, `50_back`, `100`, `200`, `500`.
2. `My_First_Project_yolov8.zip`: Roboflow YOLO export with `train/`, `valid/`, `test/`, and `data.yaml`.

The Roboflow archive is included in the project ZIP as a convenience. You can upload that copy from `dataset/positive_50_notes/`.

## Important
- The notebook uses the Roboflow labels as the positive ₹50 annotations.
- Other denominations and optional hard-negative photos are added with empty YOLO label files.
- Geometric augmentation transforms bounding boxes together with images.
- Augmentation is applied only to the training set. Validation and test images remain unaugmented.
- Images from the same capture session should stay in the same split where possible.
- Keep your existing model files as a baseline; do not replace them until the new model passes testing.

Run cells from top to bottom. Choose Runtime → Change runtime type → GPU if available.

## Output folders
After training, the final cell saves `best.pt` and `best.onnx` in `models/`, saves training/evaluation artifacts in `reports/`, and downloads a ZIP containing both folders.


## 1. Install libraries

In [ ]:
!pip -q install ultralytics albumentations opencv-python-headless pyyaml

## 2. Upload the two ZIP files

In [ ]:
from google.colab import files
uploaded = files.upload()
print("Uploaded:", list(uploaded.keys()))

Select both ZIP files. Colab may require two separate upload attempts; that is fine. Run this upload cell again to select the second file if needed.

## 3. Extract and locate datasets

In [ ]:
from pathlib import Path
import zipfile, shutil, random, os, yaml
WORK = Path("/content/smart_currency_v2")
if WORK.exists():
    shutil.rmtree(WORK)
WORK.mkdir(parents=True)

zips = list(Path("/content").glob("*.zip"))
rf_zip = next((p for p in zips if "yolov8" in p.name.lower() or "roboflow" in p.name.lower()), None)
orig_zip = next((p for p in zips if p != rf_zip), None)
if rf_zip is None or orig_zip is None:
    raise FileNotFoundError("Need both original_images.zip and the Roboflow YOLO ZIP in /content. Upload the missing ZIP and rerun this cell.")

rf_dir, orig_dir = WORK/"roboflow", WORK/"originals"
rf_dir.mkdir(); orig_dir.mkdir()
with zipfile.ZipFile(rf_zip) as z: z.extractall(rf_dir)
with zipfile.ZipFile(orig_zip) as z: z.extractall(orig_dir)

# Find the Roboflow root containing data.yaml
yaml_paths = list(rf_dir.rglob("data.yaml"))
if not yaml_paths:
    raise FileNotFoundError("Roboflow ZIP has no data.yaml.")
rf_root = yaml_paths[0].parent

# Find root that directly contains all denomination folders
denoms = {"10","20","50_front","50_back","100","200","500"}
orig_root = None
for p in [orig_dir] + [q for q in orig_dir.rglob("*") if q.is_dir()]:
    try:
        child_dirs = {q.name for q in p.iterdir() if q.is_dir()}
    except Exception:
        continue
    if denoms.issubset(child_dirs):
        orig_root = p
        break
if orig_root is None:
    raise FileNotFoundError("Could not find all seven denomination folders inside original_images.zip.")

print("Roboflow dataset:", rf_root)
print("Original images:", orig_root)

## 4. Audit the Roboflow positive labels

In [ ]:
IMG_EXTS = {".jpg",".jpeg",".png",".bmp",".webp"}
split_counts = {}
for split in ("train","valid","test"):
    imdir, lbdir = rf_root/split/"images", rf_root/split/"labels"
    imgs = [p for p in imdir.iterdir() if p.is_file() and p.suffix.lower() in IMG_EXTS] if imdir.exists() else []
    if not imgs:
        raise ValueError(f"No {split} images found at {imdir}")
    missing = [p.name for p in imgs if not (lbdir/(p.stem+".txt")).exists()]
    if missing:
        raise ValueError(f"{split}: missing label files: {missing[:8]}")
    for img in imgs:
        rows = (lbdir/(img.stem+".txt")).read_text().strip().splitlines()
        if not rows:
            raise ValueError(f"Positive image has empty label: {img.name}")
        for row in rows:
            vals = row.split()
            if len(vals) != 5:
                raise ValueError(f"Malformed YOLO row in {img.name}: {row}")
            cls, x, y, w, h = map(float, vals)
            if cls != 0 or not (0 <= x <= 1 and 0 <= y <= 1 and 0 < w <= 1 and 0 < h <= 1):
                raise ValueError(f"Invalid class/coordinates in {img.name}: {row}")
    split_counts[split] = len(imgs)
    print(f"{split}: {len(imgs)} positive images; labels verified")
print("Positive total:", sum(split_counts.values()))

## 5. Prepare the combined source splits

In [ ]:
OUT = WORK/"dataset"
if OUT.exists(): shutil.rmtree(OUT)
for split in ("train","valid","test"):
    (OUT/split/"images").mkdir(parents=True)
    (OUT/split/"labels").mkdir(parents=True)

# Copy annotated positives without changing their original split
for split in ("train","valid","test"):
    for img in (rf_root/split/"images").iterdir():
        if img.is_file() and img.suffix.lower() in IMG_EXTS:
            shutil.copy2(img, OUT/split/"images"/img.name)
            shutil.copy2(rf_root/split/"labels"/(img.stem+".txt"), OUT/split/"labels"/(img.stem+".txt"))

# Add negatives from other denominations, split 70/15/15
negative_paths=[]
for denom in ("10","20","100","200","500"):
    folder = orig_root/denom
    if not folder.exists(): raise FileNotFoundError(folder)
    negative_paths += [p for p in folder.rglob("*") if p.is_file() and p.suffix.lower() in IMG_EXTS]

# Optional extra hard negatives are not required in the initial upload.
random.seed(42)
random.shuffle(negative_paths)
n=len(negative_paths); ntr=int(.70*n); nva=int(.15*n)
negative_splits={"train":negative_paths[:ntr],
                 "valid":negative_paths[ntr:ntr+nva],
                 "test":negative_paths[ntr+nva:]}
for split, paths in negative_splits.items():
    for i, src in enumerate(paths):
        name=f"neg_{src.parent.name}_{i:05d}{src.suffix.lower()}"
        shutil.copy2(src, OUT/split/"images"/name)
        (OUT/split/"labels"/(Path(name).stem+".txt")).write_text("")

# Optional hard negatives uploaded inside a ZIP can be added in a future iteration.
data_yaml = OUT/"data.yaml"
data_yaml.write_text(
    f"path: {OUT}\ntrain: train/images\nval: valid/images\ntest: test/images\nnc: 1\nnames: ['50_rupee_note']\n"
)
for split in ("train","valid","test"):
    print(split, "images:", len(list((OUT/split/"images").glob("*"))),
          "empty labels:", sum(not p.read_text().strip() for p in (OUT/split/"labels").glob("*.txt")))

## 6. Create training-only augmentations

This uses Albumentations to transform images and bounding boxes together. It creates a few controlled variants per positive training image. Validation and test sets are not augmented.

We intentionally keep the transformations moderate so the note remains realistic and its denomination details are not destroyed.

In [ ]:
import cv2
import albumentations as A

augmenter = A.Compose(
    [
        A.RandomBrightnessContrast(brightness_limit=0.18, contrast_limit=0.18, p=0.7),
        A.Rotate(limit=12, border_mode=cv2.BORDER_CONSTANT, p=0.45),
        A.Perspective(scale=(0.02, 0.05), p=0.25),
        A.GaussianBlur(blur_limit=(3,5), p=0.15),
        A.GaussNoise(p=0.15),
    ],
    bbox_params=A.BboxParams(format="yolo", label_fields=["class_labels"], min_visibility=0.55)
)

AUG = WORK/"augmented_train"
(AUG/"images").mkdir(parents=True)
(AUG/"labels").mkdir(parents=True)

train_img_dir, train_lbl_dir = OUT/"train"/"images", OUT/"train"/"labels"
positive_train = []
for img_path in train_img_dir.iterdir():
    label_path=train_lbl_dir/(img_path.stem+".txt")
    if label_path.exists() and label_path.read_text().strip():
        positive_train.append(img_path)

aug_count=0
for img_path in positive_train:
    image=cv2.imread(str(img_path))
    if image is None: continue
    rows=[r.split() for r in (train_lbl_dir/(img_path.stem+".txt")).read_text().strip().splitlines()]
    boxes=[[float(v) for v in row[1:5]] for row in rows]
    labels=[int(float(row[0])) for row in rows]
    for j in range(2):
        result=augmenter(image=image, bboxes=boxes, class_labels=labels)
        if not result["bboxes"]: continue
        name=f"{img_path.stem}_aug{j+1}.jpg"
        cv2.imwrite(str(AUG/"images"/name), result["image"])
        lines=["0 "+" ".join(f"{v:.6f}" for v in box) for box in result["bboxes"]]
        (AUG/"labels"/(Path(name).stem+".txt")).write_text("\n".join(lines)+"\n")
        aug_count+=1

# Append generated images/labels to training split only
for p in (AUG/"images").glob("*"):
    shutil.copy2(p, train_img_dir/p.name)
    shutil.copy2(AUG/"labels"/(p.stem+".txt"), train_lbl_dir/(p.stem+".txt"))

print("Positive training originals:", len(positive_train))
print("Augmented images added:", aug_count)
print("Validation/test untouched.")

## 7. Visual check of augmented samples

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
samples=list((AUG/"images").glob("*"))[:6]
if samples:
    fig, axes=plt.subplots(2,3,figsize=(12,7))
    for ax,p in zip(axes.ravel(),samples):
        ax.imshow(Image.open(p).convert("RGB")); ax.set_title(p.name); ax.axis("off")
    for ax in axes.ravel()[len(samples):]: ax.axis("off")
    plt.tight_layout(); plt.show()
else:
    print("No augmented samples were created. Inspect labels/bounding boxes before training.")

## 8. Train YOLOv8n

In [ ]:
from ultralytics import YOLO
import torch
model=YOLO("yolov8n.pt")
train_results=model.train(
    data=str(data_yaml), epochs=80, imgsz=640,
    batch=8 if torch.cuda.is_available() else 4,
    patience=20, seed=42, pretrained=True,
    project=str(WORK/"runs"), name="smart_currency_v2",
    # Moderate built-in augmentation in addition to the offline augmentation above
    degrees=5, translate=0.08, scale=0.25, fliplr=0.0, flipud=0.0,
    hsv_h=0.01, hsv_s=0.25, hsv_v=0.20, mosaic=0.5, mixup=0.0
)

## 9. Validate and test

In [ ]:
best_path=WORK/"runs"/"smart_currency_v2"/"weights"/"best.pt"
if not best_path.exists(): raise FileNotFoundError(best_path)
best_model=YOLO(str(best_path))
val_metrics=best_model.val(data=str(data_yaml), split="val", imgsz=640)
test_metrics=best_model.val(data=str(data_yaml), split="test", imgsz=640)
print("Validation:", val_metrics)
print("Test:", test_metrics)

## 10. Export ONNX and download model

In [ ]:
# Export ONNX, then save both model formats inside the project models/ folder.
from pathlib import Path
import shutil, json, csv, zipfile
from google.colab import files

PROJECT = Path("/content/Smart-Currency-AI-V2")
MODELS_DIR = PROJECT / "models"
REPORTS_DIR = PROJECT / "reports"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

onnx_path = best_model.export(format="onnx", imgsz=640, opset=12)
onnx_path = Path(onnx_path)

# Keep the trained PyTorch model and ONNX model together in models/.
shutil.copy2(best_path, MODELS_DIR / "best.pt")
shutil.copy2(onnx_path, MODELS_DIR / "best.onnx")
print("Saved models:", MODELS_DIR / "best.pt", "and", MODELS_DIR / "best.onnx")

# Save the Ultralytics training outputs (plots, results.csv, confusion matrix, etc.).
run_dir = WORK / "runs" / "smart_currency_v2"
if run_dir.exists():
    for item in run_dir.iterdir():
        dest = REPORTS_DIR / item.name
        if item.is_dir():
            if dest.exists(): shutil.rmtree(dest)
            shutil.copytree(item, dest)
        elif item.is_file():
            shutil.copy2(item, dest)

# Save a readable summary of validation and test metrics.
def metric_summary(metrics):
    out = {}
    box = getattr(metrics, "box", None)
    if box is not None:
        for key in ("mp", "mr", "map50", "map", "maps"):
            value = getattr(box, key, None)
            if value is not None:
                try: out[key] = value.tolist() if hasattr(value, "tolist") else float(value)
                except Exception: out[key] = str(value)
    return out

summary = {
    "model": "YOLOv8n",
    "class": "50_rupee_note",
    "validation_metrics": metric_summary(val_metrics),
    "test_metrics": metric_summary(test_metrics),
    "note": "Interpret metrics cautiously; performance depends on dataset size, split quality, and fresh real-world testing."
}
(REPORTS_DIR / "evaluation_summary.json").write_text(json.dumps(summary, indent=2))
(REPORTS_DIR / "evaluation_summary.txt").write_text(
    "SMART CURRENCY AI V2 - EVALUATION SUMMARY\n\n" +
    "Validation metrics:\n" + json.dumps(summary["validation_metrics"], indent=2) +
    "\n\nTest metrics:\n" + json.dumps(summary["test_metrics"], indent=2) +
    "\n\n" + summary["note"] + "\n"
)

# Bundle models/ and reports/ so they can be downloaded together.
artifact_zip = Path("/content/Smart-Currency-AI-V2-models-and-reports.zip")
with zipfile.ZipFile(artifact_zip, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in (MODELS_DIR, REPORTS_DIR):
        for f in folder.rglob("*"):
            if f.is_file(): z.write(f, f.relative_to(PROJECT))

print("\nModels and reports are organized in:", PROJECT)
print("Download bundle:", artifact_zip)
files.download(str(artifact_zip))


## 11. Where the outputs are saved

When the export cell finishes, the notebook creates a project folder at `/content/Smart-Currency-AI-V2/` with:

- `models/best.pt` — trained Ultralytics model.
- `models/best.onnx` — exported ONNX model for deployment.
- `reports/` — copied training outputs (including `results.csv`, plots, and confusion-matrix files when generated), plus `evaluation_summary.json` and `evaluation_summary.txt`.

The last cell downloads `Smart-Currency-AI-V2-models-and-reports.zip`. Extract that ZIP into your local `Smart-Currency-AI-V2/` project folder and allow it to merge with the existing `models/` and `reports/` folders. If Windows asks whether to replace files, keep a backup of your previous model first.

## Evaluation caution

This dataset is still relatively small. A good-looking training metric does not prove robust currency recognition. Test on fresh genuine notes and many non-₹50 objects. If the test set is tiny, report that limitation honestly.
